# Phase 8: Cross-Modal Merger Test
This notebook verifies that `CrossModalMerger` links non-text chunks (image_caption, transcript) to the most semantically related text chunks.

In [ ]:
import sys
import os
sys.path.append(os.path.abspath('..'))

from src.ingestion.merging.cross_modal_merger import CrossModalMerger
from src.ingestion.embedding.embedding_pipeline import EmbeddingPipeline
print("Import successful!")

## 1. Test Cross-Modal Linking

In [ ]:
# 1. Initialize Pipeline (for embedding)
pipeline = EmbeddingPipeline("all-MiniLM-L6-v2")

# 2. Create chunks
chunks = [
    {
        "id": "text_1",
        "content": "The new solar panel efficiency reached 25% this year, a record high.",
        "modality": "text"
    },
    {
        "id": "text_2",
        "content": "Baking a chocolate cake requires flour, sugar, and cocoa powder.",
        "modality": "text"
    },
    {
        "id": "img_1",
        "content": "A graph showing solar panel efficiency climbing over the years.",
        "modality": "image_caption"
    },
    {
        "id": "vid_1",
        "content": "Now we add the cocoa powder to the mixture.",
        "modality": "transcript"
    }
]

# 3. Run Merger
merger = CrossModalMerger(similarity_threshold=0.5, embedder=pipeline.embedder)
merged_chunks = merger.merge(chunks)

# 4. Validate
img_chunk = next(c for c in merged_chunks if c["id"] == "img_1")
vid_chunk = next(c for c in merged_chunks if c["id"] == "vid_1")

print("Image relationships:", img_chunk.get("related_chunks"))
print("Video relationships:", vid_chunk.get("related_chunks"))

assert len(img_chunk["related_chunks"]) > 0
assert img_chunk["related_chunks"][0]["chunk_id"] == "text_1"
assert len(vid_chunk["related_chunks"]) > 0
assert vid_chunk["related_chunks"][0]["chunk_id"] == "text_2"

print("Cross-Modal Merger test passed!")